In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
muhammadasharib706_invoices_data_path = kagglehub.dataset_download('muhammadasharib706/invoices-data')

print('Data source import complete.')


Data source import complete.


# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# At the very top of your notebook
%pip install -q torch transformers datasets pandas Pillow numpy pytesseract seqeval tensorboard openpyxl xlsxwriter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 13.8 MB/s eta 0:00:00


In [ ]:
%pip install numpy bigframes rich gcsfs fsspec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.2 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Wed Jun 11 17:35:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# 1. Data Preparation
# ==================

# Paths
annotation_dir = "/kaggle/input/invoices-data/NER-Annotations/NER-Annotations"
image_dir = "/kaggle/input/invoices-data/NER-images/NER-images"

# Load and combine all annotation files
all_files = glob.glob(os.path.join(annotation_dir, "*.xlsx"))
dataframes = []

for file in all_files:
    df = pd.read_excel(file)

    # Standardize tags
    if 'NER_Tag' in df.columns:
        # Replace variants with standard tags
        df['NER_Tag'] = df['NER_Tag'].replace({
            'PRNO': 'PRN',
            'OQT': 'QTY',
            'GST ': 'GST',
            None: 'O',
            np.nan: 'O'
        })
        # Final cleanup
        df['NER_Tag'] = df['NER_Tag'].str.strip().fillna('O')

    # Add image path
    base_name = os.path.splitext(os.path.basename(file))[0]
    for ext in ['.jpg', '.jpeg', '.png']:
        image_path = os.path.join(image_dir, base_name + ext)
        if os.path.exists(image_path):
            df['image_path'] = image_path
            break

    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True)

# Verify required columns
assert {'Token', 'NER_Tag'}.issubset(data.columns), "Missing required columns"

# Convert all columns to string type to avoid any conversion issues
data = data.astype(str)

# 2. Label Configuration
# =====================

# Official tag set (after standardization)
labels = ["O", "PRN", "UPP", "TPP", "TB", "GST", "QTY"]
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

# 3. Dataset Preparation
# =====================

def preprocess_example(example):
    # Handle tokens - ensure we always get a list of strings
    tokens = example['Token'].split() if isinstance(example['Token'], str) else [str(t) for t in example['Token']]

    # Handle NER tags - ensure we always get valid tags
    if isinstance(example['NER_Tag'], str):
        ner_tags = []
        for tag in example['NER_Tag'].split():
            # Ensure all tags are valid
            ner_tags.append(label2id.get(tag.strip(), label2id['O']))
    else:
        tag = str(example['NER_Tag']).strip()
        ner_tags = [label2id.get(tag, label2id['O'])]

    return {
        'tokens': tokens,
        'ner_tags': ner_tags,
        'image_path': example['image_path']
    }

# Create dataset from pandas DataFrame
dataset = Dataset.from_pandas(data)

# Apply preprocessing
dataset = dataset.map(preprocess_example)

# Split dataset
dataset = dataset.train_test_split(test_size=0.2)

# Rest of your code remains the same...
# Rest of your code remains the same...
# 4. Model Setup
# ==============

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

def tokenize_and_align(examples):
    try:
        image = Image.open(examples['image_path']).convert("RGB")
    except:
        # Create blank image if file missing
        image = Image.new('RGB', (224, 224), (255, 255, 255))

    # Get tokens and labels
    tokens = examples['tokens']
    labels = examples['ner_tags']

    # Ensure tokens and labels have the same length
    if len(tokens) != len(labels):
        # Truncate to the shorter length
        min_length = min(len(tokens), len(labels))
        tokens = tokens[:min_length]
        labels = labels[:min_length]

    # Create dummy boxes (all tokens get the same box for simplicity)
    boxes = [[0, 0, 1000, 1000]] * len(tokens)

    # Tokenize
    encoding = processor(
        image,
        text=tokens,
        boxes=boxes,
        word_labels=labels,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    return {
        'input_ids': encoding['input_ids'].squeeze(),
        'attention_mask': encoding['attention_mask'].squeeze(),
        'bbox': encoding['bbox'].squeeze(),
        'pixel_values': encoding['pixel_values'].squeeze(),
        'labels': encoding['labels'].squeeze()
    }

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_and_align, batched=False)
# 5. Training Setup
# =================
# 5. Training Setup
# =================

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)
# Updated training arguments compatible with newer Transformers versions
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",  # Changed from evaluation_strategy to eval_strategy
    eval_steps=200,
    save_steps=200,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_accumulation_steps=2,
    report_to="none"
)
data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=processor,
    data_collator=data_collator,
)
# 6. Training
# ===========
trainer.train()
trainer.save_model("best_model")
processor.save_pretrained("best_model")

Map:   0%|          | 0/13322 [00:00<?, ? examples/s]

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss
200,0.721900,0.688268
400,0.571000,0.567384
600,0.519300,0.482046
800,0.501100,0.462728


Step,Training Loss,Validation Loss
200,0.721900,0.688268
400,0.571000,0.567384
600,0.519300,0.482046
800,0.501100,0.462728
1000,0.465500,0.440432
1200,0.410700,0.428856
1400,0.447500,0.424616
1600,0.329500,0.416648
1800,0.341800,0.411736
2000,0.434100,0.385635


In [ ]:
import transformers
print(transformers.__version__)

4.51.3
